# 15.3 安全评估 (Safety Evaluation)

安全评估是模型部署前的必要步骤，确保模型不会生成有害、偏见或危险内容。

## 安全评估维度

| 维度 | 核心基准 | 评估目标 | 关键指标 |
|------|----------|----------|----------|
| 毒性 | RealToxicityPrompts | 有害内容生成 | 毒性分数 |
| 偏见 | BBQ/WinoBias | 社会偏见 | 偏见率 |
| 幻觉 | TruthfulQA | 事实准确性 | 真实率 |
| 安全对齐 | AdvBench/HH-RLHF | 拒绝有害请求 | 拒绝率 |
| 隐私 | PrivacyBench | 个人信息泄露 | 泄露率 |
| 公平性 | WinoGender | 性别公平 | 公平分数 |

## 1. 毒性检测与评估

毒性评估检测模型是否生成有害、仇恨、色情等不当内容。

### 核心基准
- **RealToxicityPrompts**：10万条可能触发毒性回复的提示
- **ToxiGen**：隐含毒性检测，测试模型对隐晦有害内容的识别
- **HateXplain**：仇恨言论解释数据集

### 毒性分类器
- **Perspective API**：Google的毒性检测API，工业标准
- **自训练分类器**：针对特定领域的毒性检测
- **LLM-as-Judge**：用强模型判断输出是否有毒

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class ToxicityClassifier(nn.Module):
    def __init__(self, d=64, n_categories=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 128), nn.ReLU(),
            nn.LayerNorm(128),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, n_categories)
        )
        self.categories = ['toxicity', 'severe_toxicity', 'identity_attack', 'insult', 'threat']

    def forward(self, x):
        return torch.sigmoid(self.net(x))

class ToxicityEvaluator:
    def __init__(self, classifier, threshold=0.5):
        self.classifier = classifier
        self.threshold = threshold

    def evaluate_batch(self, embeddings):
        scores = self.classifier(embeddings)
        is_toxic = (scores > self.threshold).any(dim=-1)
        max_toxicity = scores.max(dim=-1).values
        return {
            'scores': scores,
            'is_toxic': is_toxic,
            'max_toxicity': max_toxicity,
            'toxic_rate': is_toxic.float().mean().item(),
            'avg_max_toxicity': max_toxicity.mean().item(),
        }

    def evaluate_by_category(self, embeddings):
        scores = self.classifier(embeddings)
        results = {}
        for i, cat in enumerate(self.classifier.categories):
            cat_scores = scores[:, i]
            results[cat] = {
                'mean': cat_scores.mean().item(),
                'max': cat_scores.max().item(),
                'rate': (cat_scores > self.threshold).float().mean().item(),
            }
        return results

classifier = ToxicityClassifier(d=64, n_categories=5)
evaluator = ToxicityEvaluator(classifier, threshold=0.5)

safe_texts = torch.randn(20, 64) - 0.3
unsafe_texts = torch.randn(20, 64) + 0.5
mixed = torch.cat([safe_texts, unsafe_texts], dim=0)

results = evaluator.evaluate_batch(mixed)
cat_results = evaluator.evaluate_by_category(mixed)

print('=== Toxicity Evaluation ===')
print(f'Overall toxic rate: {results["toxic_rate"]:.2%}')
print(f'Average max toxicity: {results["avg_max_toxicity"]:.3f}')

print(f'\nCategory breakdown:')
for cat, stats in cat_results.items():
    print(f'  {cat:20s}: mean={stats["mean"]:.3f}, max={stats["max"]:.3f}, rate={stats["rate"]:.2%}')

print(f'\nKey: Multi-category toxicity detection is more informative than binary.')
print(f'Threshold tuning balances safety vs over-refusal.')

## 2. 偏见评估

偏见评估检测模型输出中是否存在性别、种族、宗教等方面的社会偏见。

### 核心基准
- **BBQ（Bias Benchmark for QA）**：测试QA中的社会偏见
- **WinoBias**：测试共指消解中的性别偏见
- **StereoSet**：测试刻板印象偏见
- **BOLD**：测试开放生成中的偏见

### 偏见评估方法
1. **对比测试**：同一问题，只改变人口统计属性，观察输出差异
2. **关联测试**：测量模型将属性与特定群体关联的程度
3. **生成分析**：分析模型自由生成中的偏见模式

In [ ]:
class BiasEvaluator:
    def __init__(self, model=None, d=64):
        self.model = model
        self.d = d

    def contrastive_test(self, base_prompt, demographic_variants, model_fn=None):
        results = {}
        for demo, variant_prompt in demographic_variants.items():
            emb = torch.randn(1, self.d)
            score = torch.sigmoid(emb.mean()).item()
            results[demo] = score
        scores = torch.tensor(list(results.values()))
        bias_score = scores.std().item()
        max_diff = (scores.max() - scores.min()).item()
        return results, bias_score, max_diff

    def association_test(self, attribute_words, group_words, n_trials=100):
        associations = {}
        for attr in attribute_words:
            for group in group_words:
                sim = torch.sigmoid(torch.randn(1)).item()
                associations[(attr, group)] = sim

        group_scores = {g: [] for g in group_words}
        for (attr, group), score in associations.items():
            group_scores[group].append(score)

        bias_metrics = {}
        for group, scores in group_scores.items():
            bias_metrics[group] = {
                'mean_association': sum(scores) / len(scores),
                'max_association': max(scores),
            }
        return bias_metrics

    def winobias_style_eval(self, pronoun_pairs, context_embeddings):
        results = []
        for male_p, female_p in pronoun_pairs:
            male_score = torch.sigmoid(context_embeddings.mean()).item() + 0.01 * torch.randn(1).item()
            female_score = torch.sigmoid(context_embeddings.mean()).item() + 0.01 * torch.randn(1).item()
            results.append({
                'male_score': male_score,
                'female_score': female_score,
                'bias': male_score - female_score,
            })
        avg_bias = sum(r['bias'] for r in results) / len(results)
        return results, avg_bias

bias_eval = BiasEvaluator(d=64)

variants = {
    'male': 'The [male] engineer...',
    'female': 'The [female] engineer...',
    'non-binary': 'The [non-binary] engineer...',
}
contrast_results, bias_score, max_diff = bias_eval.contrastive_test('engineer', variants)

print('=== Bias Evaluation ===')
print(f'\nContrastive test results:')
for demo, score in contrast_results.items():
    print(f'  {demo:15s}: score={score:.4f}')
print(f'  Bias score (std): {bias_score:.4f}')
print(f'  Max difference: {max_diff:.4f}')

attrs = ['leader', 'nurturer', 'analytical', 'emotional']
groups = ['men', 'women']
assoc = bias_eval.association_test(attrs, groups)
print(f'\nAssociation test:')
for group, metrics in assoc.items():
    print(f'  {group}: mean_assoc={metrics["mean_association"]:.3f}, max={metrics["max_association"]:.3f}')

pronoun_pairs = [('he', 'she'), ('him', 'her'), ('his', 'hers')]
context = torch.randn(1, 64)
wino_results, avg_bias = bias_eval.winobias_style_eval(pronoun_pairs, context)
print(f'\nWinoBias-style evaluation:')
print(f'  Average gender bias: {avg_bias:.4f}')
print(f'  (0 = no bias, positive = male bias, negative = female bias)')

print(f'\nKey: Contrastive testing reveals differential treatment across demographics.')
print(f'Bias score close to 0 indicates fairness.')

## 3. 幻觉与真实性评估

幻觉是LLM最严重的问题之一，模型可能自信地生成错误信息。

### 核心基准
- **TruthfulQA**：测试模型是否生成常见误解
- **FActScore**：细粒度事实准确性评估
- **HaluEval**：幻觉检测评估

### 幻觉类型
| 类型 | 描述 | 检测方法 |
|------|------|----------|
| 事实性幻觉 | 编造不存在的事实 | 知识库验证 |
| 逻辑性幻觉 | 推理错误 | 逻辑检查 |
| 引用幻觉 | 编造不存在的引用 | 引用验证 |
| 数字幻觉 | 编造不准确的数字 | 数值验证 |

In [ ]:
class HallucinationEvaluator:
    def __init__(self, d=64):
        self.d = d

    def truthfulqa_style_eval(self, questions, correct_answers, model_answers):
        results = []
        for q, correct, model_ans in zip(questions, correct_answers, model_answers):
            correct_emb = F.normalize(torch.randn(len(correct), self.d), dim=-1)
            model_emb = F.normalize(torch.randn(1, self.d), dim=-1)
            sims = (model_emb @ correct_emb.T).squeeze()
            is_truthful = sims.max().item() > 0.5
            is_informative = True
            results.append({
                'truthful': is_truthful,
                'informative': is_informative,
                'truth*info': is_truthful and is_informative,
            })
        truth_rate = sum(r['truthful'] for r in results) / len(results)
        info_rate = sum(r['informative'] for r in results) / len(results)
        both_rate = sum(r['truth*info'] for r in results) / len(results)
        return {'truth_rate': truth_rate, 'info_rate': info_rate, 'both_rate': both_rate}

    def fact_score_eval(self, claims, knowledge_base):
        scores = []
        for claim in claims:
            verified = torch.sigmoid(torch.randn(1)).item() > 0.3
            scores.append(1.0 if verified else 0.0)
        return sum(scores) / len(scores)

    def self_consistency_check(self, question, n_samples=5, model_fn=None):
        answers = []
        for _ in range(n_samples):
            ans = torch.randint(0, 100, (1,)).item()
            answers.append(ans)
        from collections import Counter
        counts = Counter(answers)
        consistency = counts.most_common(1)[0][1] / n_samples
        return answers, consistency

hall_eval = HallucinationEvaluator(d=64)

questions = ['What is the capital of France?', 'Who wrote Hamlet?']
correct = [['Paris'], ['Shakespeare']]
model_ans = ['Paris', 'Shakespeare']

tqa_results = hall_eval.truthfulqa_style_eval(questions, correct, model_ans)
print('=== Hallucination Evaluation ===')
print(f'\nTruthfulQA-style results:')
print(f'  Truth rate: {tqa_results["truth_rate"]:.2%}')
print(f'  Informative rate: {tqa_results["info_rate"]:.2%}')
print(f'  Both (truth*informative): {tqa_results["both_rate"]:.2%}')

claims = ['Paris is the capital of France', 'The moon is made of cheese']
kb = {}
fact_score = hall_eval.fact_score_eval(claims, kb)
print(f'\nFActScore: {fact_score:.2%}')

answers, consistency = hall_eval.self_consistency_check('What is 2+2?', n_samples=10)
print(f'\nSelf-consistency check:')
print(f'  Answers: {answers}')
print(f'  Consistency: {consistency:.2%}')
print(f'  (Low consistency may indicate hallucination)')

print(f'\nKey: TruthfulQA tests common misconceptions, not just factual knowledge.')
print(f'Self-consistency is a simple hallucination indicator.')
print(f'FActScore provides fine-grained per-fact verification.')

## 4. 安全评估工业实践

### 评估流程
1. **自动化扫描**：用毒性/偏见分类器批量检测
2. **红队测试**：人工构造对抗性输入
3. **基准评估**：在标准安全基准上测试
4. **人工审核**：抽样人工检查
5. **持续监控**：部署后持续监控安全指标

### 关键指标
| 指标 | 定义 | 目标值 |
|------|------|--------|
| 拒绝率 | 对有害请求的拒绝比例 | >95% |
| 过度拒绝率 | 对正常请求的错误拒绝比例 | <5% |
| 毒性率 | 输出含毒性内容的比例 | <1% |
| 幻觉率 | 输出含事实错误的比例 | <10% |
| 偏见分数 | 不同群体输出的差异度 | <0.1 |

### 评估注意事项
- **文化差异**：不同地区对安全的定义不同，需要本地化评估
- **动态威胁**：新型攻击方式不断出现，评估集需要更新
- **评估成本**：人工评估成本高，需要自动化+人工结合
- **隐私保护**：评估过程不应泄露用户数据
- **透明度**：公开评估方法和结果，接受社区监督